# Phase 3: Baseline Model & UI (Esencial)

**Goals:** Safe vs Toxic baselines, overfitting check, FastAPI `/predict`, React Watch Page UI.

## Environment (`uv`)

```bash
uv sync
uv run python -m src.pipeline.phase3_train_baseline
uv run uvicorn src.api.main:app --reload --port 8000
# other terminal:
cd frontend && npm install && npm run dev
```

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.pipeline.phase3_train_baseline import train_phase3
from src.api.inference import predict_comment
from src.utils.execution_log import log_event

LOG_PATH = ROOT / "logs/execution.log"
log_event("Phase 3 notebook started", phase="3", log_path=LOG_PATH)

## 1. Train baselines & overfitting report

In [2]:
report = train_phase3()
print(json.dumps(report, indent=2))

{
  "phase": 3,
  "model_path": "models/phase3_baseline.joblib",
  "best_model": "logistic_regression",
  "selection_reason": "best test F1 (no model met 5% overfitting gap \u2014 increase regularization in Phase 4)",
  "passes_overfitting": false,
  "metrics": {
    "accuracy": {
      "metric": "accuracy",
      "train": 0.8925,
      "test": 0.735,
      "gap_pct": 15.75,
      "passes": false
    },
    "f1_toxic": {
      "metric": "f1_toxic",
      "train": 0.8822,
      "test": 0.7072,
      "gap_pct": 17.5,
      "passes": false
    },
    "passes_overfitting": false
  },
  "all_models": [
    {
      "name": "naive_bayes",
      "test_f1_toxic": 0.6705,
      "passes_overfitting": false,
      "metrics": {
        "accuracy": {
          "metric": "accuracy",
          "train": 0.9487,
          "test": 0.715,
          "gap_pct": 23.38,
          "passes": false
        },
        "f1_toxic": {
          "metric": "f1_toxic",
          "train": 0.9438,
          "test": 0.670

## Evaluation (train vs test, 5% gap rule)

In [3]:
from IPython.display import display

from src.evaluation.report_display import (
    overall_pass_label,
    print_evaluation_banner,
    report_to_evaluation_df,
)

print_evaluation_banner("Phase 3")
eval_df = report_to_evaluation_df(report, model_name=report["best_model"])
display(eval_df)

overall = {
    "model": report["best_model"],
    "overall_pass": overall_pass_label(report),
    "passes_overfitting": report["passes_overfitting"],
    "selection_reason": report["selection_reason"],
}
print(json.dumps(overall, indent=2))

Phase 3 — OVERFITTING EVALUATION
Rule: |train − test| < 5 percentage points (accuracy & F1 toxic)


,model,metric,train,test,gap_pp,pass
0,logistic_regression,accuracy,0.8925,0.7350,15.75,FAIL
1,logistic_regression,f1_toxic,0.8822,0.7072,17.50,FAIL
2,naive_bayes,accuracy,0.9487,0.7150,23.38,FAIL
3,naive_bayes,f1_toxic,0.9438,0.6705,27.32,FAIL
4,logistic_regression,accuracy,0.8925,0.7350,15.75,FAIL
5,logistic_regression,f1_toxic,0.8822,0.7072,17.50,FAIL
6,random_forest,accuracy,0.8113,0.7000,11.13,FAIL
7,random_forest,f1_toxic,0.7736,0.6250,14.86,FAIL


{
  "model": "logistic_regression",
  "overall_pass": "FAIL",
  "passes_overfitting": false,
  "selection_reason": "best test F1 (no model met 5% overfitting gap \u2014 increase regularization in Phase 4)"
}


## 2. Inference smoke test

In [4]:
for sample in ["Thanks for the great video!", "You are awful and should disappear"]:
    r = predict_comment(sample)
    print(sample[:50], "... ->", r)

Thanks for the great video! ... -> {'toxic_score': 48.65, 'label': 'Safe', 'status_color': 'yellow', 'mode': 'binary', 'version': 'phase3-esencial', 'model_name': 'random_forest_tuned'}
You are awful and should disappear ... -> {'toxic_score': 49.67, 'label': 'Safe', 'status_color': 'yellow', 'mode': 'binary', 'version': 'phase3-esencial', 'model_name': 'random_forest_tuned'}


## Conclusion

Phase 3 Esencial: binary Safe/Toxic model saved, FastAPI + React UI wired. See KEY FINDINGS below.

In [5]:
print("=" * 60)
print("PHASE 3 — KEY FINDINGS")
print("=" * 60)
print(f"Best model: {report['best_model']}")
print(f"Selection: {report['selection_reason']}")
print(f"Overfitting pass: {report['passes_overfitting']}")
m = report['metrics']
print(f"Test F1 (toxic): {m['f1_toxic']['test']} | gap: {m['f1_toxic']['gap_pct']}%")
print(f"Test accuracy: {m['accuracy']['test']} | gap: {m['accuracy']['gap_pct']}%")
print(f"Model: {ROOT / report['model_path']}")
print("API: uv run uvicorn src.api.main:app --reload")
print("UI:  cd frontend && npm run dev  (proxy -> :8000)")
print("=" * 60)
log_event("Phase 3 notebook completed", phase="3", log_path=LOG_PATH)

PHASE 3 — KEY FINDINGS
Best model: logistic_regression
Selection: best test F1 (no model met 5% overfitting gap — increase regularization in Phase 4)
Overfitting pass: False
Test F1 (toxic): 0.7072 | gap: 17.5%
Test accuracy: 0.735 | gap: 15.75%
Model: /Users/miraekang/proyectos/ai-nlp/models/phase3_baseline.joblib
API: uv run uvicorn src.api.main:app --reload
UI:  cd frontend && npm run dev  (proxy -> :8000)
